# AI Resume Screening: Data-Science Workflow

This notebook is a beginner-friendly learning companion for the AI Resume Screening project. It is not a second Streamlit application and it does not make hiring decisions.

All candidate results below are **synthetic demonstration data**, included only to show the workflow without using real applicant or hiring data.

## 1. Import project tools

Pandas provides tables, NumPy provides numerical operations, Matplotlib provides screening/ranking charts, and scikit-learn provides the small TF-IDF and cosine-similarity demonstrations.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.results_analysis import calculate_score_statistics, create_candidate_dataframe
from src.visualizations import create_overall_score_chart, create_similarity_skill_match_chart

## 2. Load demonstration candidate results

The application produces candidate dictionaries after screening resumes. These **synthetic demonstration results** have the same shape. Their overall scores follow the existing formula: `0.40 * resume_similarity + 0.60 * skill_match`.

In [ ]:
synthetic_candidate_results = [
    {'filename': 'sample_candidate_a.pdf', 'resume_similarity': 70.0, 'skill_match': 80.0, 'overall_score': 76.0},
    {'filename': 'sample_candidate_b.pdf', 'resume_similarity': 55.0, 'skill_match': 60.0, 'overall_score': 58.0},
    {'filename': 'sample_candidate_c.pdf', 'resume_similarity': 45.0, 'skill_match': 40.0, 'overall_score': 42.0},
]
synthetic_candidate_results

## 3. Create a Pandas DataFrame and rank candidates

A DataFrame is a table with rows and columns. The project helper selects candidate scores, sorts from highest to lowest overall screening score, and adds a rank. This is a screening order, not a hiring decision.

In [ ]:
candidate_dataframe = create_candidate_dataframe(synthetic_candidate_results)
candidate_dataframe

## 4. Calculate NumPy statistics

NumPy works with numeric arrays. We calculate the mean, minimum, maximum, and population standard deviation of the overall screening scores.

In [ ]:
overall_scores = candidate_dataframe['Overall Score'].to_numpy(dtype=float)
print('Overall scores:', overall_scores)
print(f'Mean: {np.mean(overall_scores):.2f}')
print(f'Minimum: {np.min(overall_scores):.2f}')
print(f'Maximum: {np.max(overall_scores):.2f}')
print(f'Standard deviation: {np.std(overall_scores):.2f}')
calculate_score_statistics(candidate_dataframe)

## 5. Visualize screening and ranking results with Matplotlib

These reusable project functions visualize the candidate scores already calculated. They are **candidate screening/ranking visualizations, not accuracy charts**.

In [ ]:
overall_score_figure = create_overall_score_chart(candidate_dataframe)
plt.show()
comparison_figure = create_similarity_skill_match_chart(candidate_dataframe)
plt.show()

## 6. Demonstrate TF-IDF

TF-IDF converts words into numeric features. Term Frequency considers how often a word appears; Inverse Document Frequency reduces the weight of words common to many documents. This is a small teaching example, not resume data or a trained model.

In [ ]:
example_resume = 'python pandas sql'
example_job_description = 'python sql machine learning'
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform([example_resume, example_job_description])
tfidf_dataframe = pd.DataFrame(tfidf_matrix.toarray(), index=['Example resume', 'Example job description'], columns=vectorizer.get_feature_names_out())
tfidf_dataframe

## 7. Demonstrate cosine similarity

Cosine similarity compares the direction of two TF-IDF vectors. A higher value means the texts share more important terms in this representation. The application multiplies the 0-to-1 result by 100 to display resume similarity as a percentage.

In [ ]:
similarity_value = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
print(f'Cosine similarity: {similarity_value:.4f}')
print(f'Resume similarity percentage: {similarity_value * 100:.2f}%')

## 8. Understand the four score terms

| Term | Meaning in this project | What it is not |
| --- | --- | --- |
| Resume similarity | TF-IDF cosine similarity between resume text and job-description text, shown as a percentage. | A prediction of candidate quality. |
| Skill match | Percentage of recognized job-description skills also found in the resume. | A complete measure of all qualifications. |
| Weighted screening score | `0.40 * resume similarity + 0.60 * skill match`; used to rank screening results. | Accuracy or a hiring decision. |
| Accuracy | A metric comparing predictions with known labels. | The weighted screening score in this project. |

The current workflow uses a weighted screening heuristic. It does not use a classification model or calculate accuracy.

## 9. Summary

Candidate-result dictionaries become a ranked Pandas DataFrame, NumPy summarizes scores, Matplotlib visualizes screening signals, and TF-IDF plus cosine similarity explains the text-comparison signal. The Streamlit app remains the user interface; this notebook is only a learning and analysis companion.

## 10. Evaluation Metrics - Separate Demonstration

This is a separate evaluation experiment using the synthetic/manual labeled dataset in `data/evaluation/sample_labeled_results.csv`. It does not change how resumes are analyzed or how the existing 40% similarity + 60% skill-match screening score is calculated.

We choose a threshold, turn scores into `suitable` or `not_suitable` predictions, and compare those predictions with manually assigned ground-truth labels.

### Choose a threshold and create predictions

A threshold is a decision rule, not a new trained model. In this demonstration, a score of 60 or higher is predicted as `suitable`; a lower score is predicted as `not_suitable`.

In [ ]:
from src.evaluation import DEFAULT_SCREENING_THRESHOLD, add_threshold_predictions, calculate_threshold_metrics, load_evaluation_dataset

evaluation_dataframe = load_evaluation_dataset()
evaluation_predictions = add_threshold_predictions(evaluation_dataframe, threshold=DEFAULT_SCREENING_THRESHOLD)
evaluation_predictions[['candidate', 'screening_score', 'actual_label', 'predicted_label']]

### Calculate and interpret the metrics

- **Accuracy:** how many predictions were correct overall.
- **Precision:** of the candidates predicted `suitable`, how many were actually suitable.
- **Recall:** of the candidates actually suitable, how many were identified as suitable.
- **F1-score:** a combined measure of precision and recall.

These metrics evaluate the threshold-based decisions against manually labeled ground truth. They do **not** mean that the existing 40% similarity + 60% skill-match score is 40% accurate or 60% accurate. The CSV is synthetic/manual demonstration data, not a real-world validation dataset.

In [ ]:
threshold_metrics = calculate_threshold_metrics(evaluation_predictions)
threshold_metrics

## 11. Generative AI - Optional Candidate Feedback

Generative AI uses a large language model (LLM) to produce natural-language text. In this project, the LLM is used only to explain an already-computed screening result and suggest practical resume improvements.

The existing TF-IDF and cosine-similarity workflow compares text numerically. Skill matching checks the known skill list, and the deterministic 40% similarity + 60% skill-match formula creates the screening score. The LLM does not replace or recalculate any of those values.

The application passes structured fields such as matched skills, missing skills, similarity, skill-match, and overall screening scores to the optional feedback module. API keys must never be hardcoded; the application reads `OPENAI_API_KEY` from the environment.

### Mock structured feedback request

This mock example shows the shape of information that could be sent to an LLM. It does not import the OpenAI SDK, require an API key, or make a network call.

In [ ]:
mock_feedback_request = {
    'candidate': 'synthetic_candidate.pdf',
    'matched_skills': ['python', 'pandas'],
    'missing_skills': ['sql'],
    'resume_similarity': 70.0,
    'skill_match': 80.0,
    'overall_screening_score': 76.0,
}

mock_feedback = (
    'Mock feedback: The candidate shows the supplied Python and Pandas strengths. '
    'The supplied SQL gap is worth addressing with concrete resume evidence.'
)

mock_feedback_request, mock_feedback